# hidden_bench

> Phase 13: HiddenBench's 65 hidden-profile tasks (Li, Naito, Shirado, ICML 2026; MIT licence, vendored in `data/hiddenbench/`) in the vote's format, so the Track A fixes run on an established benchmark: tasks with any number of options, the task's own description, an ungrouped fact table and the one-agent ceiling.

In [ ]:
#| default_exp hidden_bench

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from pathlib import Path

import numpy as np

from explRL.hidden_profile import DISSENT, ProfileEnv

import explRL

HB_PATH = Path(explRL.__file__).resolve().parents[1] / 'data' / 'hiddenbench' / 'benchmark.json'   # the repo's data/


def hbItems(path=HB_PATH):
    "HiddenBench's 65 tasks (the vendored, unchanged benchmark.json)."
    return json.loads(Path(path).read_text())


def hbTask(item, seed, N=4):
    """One HiddenBench task in hidden_profile's task format: options in a shuffled order (`best` = the correct one), the
    shared facts held by all, hidden fact j held by agent j (with fewer hidden facts than agents, the last agents hold only
    shared facts), each agent's facts in its own shuffled order."""
    rng = np.random.default_rng(seed)
    names = [str(x) for x in rng.permutation(item['possible_answers'])]
    fact = lambda text, holders: dict(option=None, sign=0, text=str(text), holders=holders, private=len(holders) < N,
                                      ruleOut=False)
    facts = [fact(x, list(range(N))) for x in item['shared_information']]
    facts += [fact(x, [j]) for j, x in enumerate(item['hidden_information'][:N])]
    facts = [facts[k] for k in rng.permutation(len(facts))]
    held = [[k for k, f in enumerate(facts) if i in f['holders']] for i in range(N)]
    held = [[h[k] for k in rng.permutation(len(h))] for h in held]
    return dict(seed=seed, variant='hb', domain=f"hiddenbench:{item['id']}", N=N, names=names,
                best=names.index(str(item['correct_answer'])), facts=facts, held=held, noun='option',
                question=item['description'], item=item['id'])


def _choices(names):
    return ', '.join(names[:-1]) + ' or ' + names[-1]


class HBEnv(ProfileEnv):
    "ProfileEnv for HiddenBench tasks: the task's own description, any number of options, an ungrouped fact table."
    def rules(self):
        t = self.task
        s = (f"{t['question']}\n\nHow this discussion works: you are one of {t['N']} members. In each of {self.rounds} rounds "
             "every member posts once to a shared board: either one of their facts (kind 'fact', fact = its id, for example "
             "'F2'; the board shows that fact's exact sentence) or an opinion (kind 'opinion', text = up to 300 characters). "
             f"Posts appear on the board after the round. After the last round every member votes privately for "
             f"{_choices(t['names'])}; the option with a majority of the votes is chosen.")
        if self.rule == 'factsFirst':
            s += ' In round 1 only facts may be posted: an opinion in round 1 is rejected.'
        if self.table:
            s += ' The board lists each fact posted so far once, without who posted it; opinions are listed below.'
        if self.hint == 'dissent':
            s += ' ' + DISSENT
        return s + (" Reply with one JSON object: kind, fact (the fact's id, or '' for an opinion), text (the opinion, "
                    "or '' for a fact) and notes (private notes to yourself, shown to you next round).")

    def boardText(self):
        if not self.table:
            return super().boardText()
        posted = sorted({p['fact'] for p in self.board if p['kind'] == 'fact'})
        fs = [self.task['facts'][k]['text'] for k in posted]
        ops = [f"round {p['round']}, agent {p['agent']}: {p['text']}" for p in self.board if p['kind'] == 'opinion']
        return '\n'.join(['Facts posted so far:'] + (['- ' + x for x in fs] or ['none.']) + ['Opinions:'] + (ops or ['none.']))

    def voteText(self, i):
        return (f'The discussion is over. You are agent {i}.\n{self.ownFacts(i)}\n{self.boardText()}\n'
                f"Vote privately for {_choices(self.task['names'])}: reply with one JSON object, vote and notes.")


def hbSolo(task):
    "The one-agent ceiling prompt: the description, every fact, then the vote."
    facts = '\n'.join('- ' + f['text'] for f in task['facts'])
    return (f"{task['question']}\n\nYou decide alone. These are all the facts known about the options:\n{facts}\n"
            f"Vote for {_choices(task['names'])}: reply with one JSON object, vote and notes.")


Tests: the tasks convert faithfully, the environment handles any number of options, and the runner plays HiddenBench cells.

In [ ]:
from collections import Counter
from explRL.hidden_profile import voteSchema
items = hbItems()
assert len(items) == 65 and all(x['correct_answer'] in x['possible_answers'] for x in items)
for it in items:
    t = hbTask(it, 7)
    assert t['names'][t['best']] == it['correct_answer'] and sorted(t['names']) == sorted(it['possible_answers'])
    texts = lambda i: {t['facts'][k]['text'] for k in t['held'][i]}
    for i in range(4):          # each agent: every shared fact plus its own hidden one (if any), verbatim
        own = set(it['shared_information']) | ({it['hidden_information'][i]} if i < len(it['hidden_information']) else set())
        assert texts(i) == own, (it['id'], i)
    assert sum(f['private'] for f in t['facts']) == len(it['hidden_information'])
assert len({tuple(hbTask(items[0], s)['names']) for s in range(30)}) > 1          # option order varies with the seed
assert hbTask(items[0], 3) == hbTask(items[0], 3)
# the environment: description, every option, an ungrouped table, K-option votes
t = hbTask(items[0], 1)
env = HBEnv(t, rounds=2, table=True, hint='dissent')
r = env.rules()
assert items[0]['description'][:60] in r and all(n in r for n in t['names']) and 'without who posted it' in r and DISSENT in r
for i in range(4):
    assert env.post(i, dict(kind='fact', fact='F1', text='', notes=''))
env.endRound()
b = env.boardText()
assert b.startswith('Facts posted so far:') and len([l for l in b.split('\n') if l.startswith('- ')]) == len({p['fact'] for p in env.board})
assert all(n in env.voteText(0) for n in t['names'])
assert voteSchema(t['names'])['properties']['vote']['enum'] == t['names'] and len(t['names']) in (3, 4)
assert env.vote(0, dict(vote=t['names'][t['best']], notes='')) == t['best'] and env.success([t['best']] * 3 + [None])
s = hbSolo(t)
assert all(f['text'] in s for f in t['facts']) and items[0]['description'][:60] in s

In [ ]:
# through runProfile with the fake client: HiddenBench cells, every arm and the solo ceiling
import tempfile
from pathlib import Path
from explRL.bench_grid import readCSV, writeCSV
from explRL.llm_fake import FakeClient, fakeReply
from explRL.llm_client import LLMClient, ResponseCache
from explRL.llm_grid import HAIKU
from explRL.profile_runs import profileCell, runProfile
cells = [profileCell('hbTest', a, 'hb', 10 * i + 1, bench='hiddenbench', item=i) for a in ('baseline', 'table', 'solo') for i in (1, 2)]
with tempfile.TemporaryDirectory() as d:
    mk = lambda: LLMClient(HAIKU, 'low', 8000, ResponseCache(Path(d) / 'c'), None, client=FakeClient(fakeReply), timing=[])
    rows = runProfile(cells, mk(), Path(d) / 'logs')
    assert len(rows) == 6 and {r['domain'] for r in rows} == {'hiddenbench:1', 'hiddenbench:2'}
    assert all(r['calls'] == (1 if r['level'].startswith('solo') else 16) for r in rows)
    again = runProfile(cells, mk())
    assert [r['success'] for r in again] == [r['success'] for r in rows]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()